# ♻️ WasteVision Studio — 02: Dataset Inspection & OpenCV Preprocessing
### فحص مجموعة بيانات TrashNet والمعالجة البصرية المتقدمة بـ OpenCV
يتضمن جرد الصور، فحص التوزيع التكراري للفئات، تطبيق تحويلات الألوان (RGB, HSV, LAB CLAHE)، ورسم مجسم سطح الحواف ثلاثي الأبعاد باستخدام Sobel.


In [ ]:
import sys, os, urllib.request, zipfile
from pathlib import Path
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt
import cv2
import plotly.graph_objects as go
from PIL import Image

# 1. تنزيل وفك مجموعة TrashNet إذا لم تكن موجودة
DATA_ROOT = Path("data")
DATA_DIR = DATA_ROOT / "dataset" / "dataset-resized"
if not DATA_DIR.exists():
    DATA_DIR = DATA_ROOT / "dataset"

print("مجلد البيانات المستهدف:", DATA_DIR)


In [ ]:
# 2. جرد الصور والفئات
classes = sorted([d.name for d in DATA_DIR.iterdir() if d.is_dir() and not d.name.startswith(".")])
records = []
for c in classes:
    for f in (DATA_DIR / c).glob("*.*"):
        if f.suffix.lower() in [".jpg", ".jpeg", ".png", ".webp", ".bmp"]:
            records.append((str(f), c))

print(f"الفئات المكتشفة ({len(classes)}):", classes)
print(f"إجمالي الصور الصالحة:", len(records))

counts = Counter([r[1] for r in records])
for c in classes:
    print(f"  • {c:12s}: {counts[c]} صورة")


In [ ]:
# 3. رسم مخطط توزيع الفئات
fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(classes, [counts[c] for c in classes], color="#0c9b79")
ax.set_title("TrashNet Class Distribution")
ax.set_ylabel("Number of Images")
plt.xticks(rotation=20)
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# 4. معالجة عينة بـ OpenCV: RGB vs HSV vs CLAHE
sample_file = records[0][0]
bgr = cv2.imread(sample_file)
rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
hsv = cv2.cvtColor(bgr, cv2.COLOR_BGR2HSV)
hue = hsv[:, :, 0]

lab = cv2.cvtColor(bgr, cv2.COLOR_BGR2LAB)
l, a, b = cv2.split(lab)
clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
enhanced_rgb = cv2.cvtColor(cv2.merge([clahe.apply(l), a, b]), cv2.COLOR_LAB2RGB)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(rgb); axes[0].set_title("Original RGB"); axes[0].axis("off")
axes[1].imshow(hue, cmap="hsv"); axes[1].set_title("HSV Hue Channel"); axes[1].axis("off")
axes[2].imshow(enhanced_rgb); axes[2].set_title("CLAHE Enhanced"); axes[2].axis("off")
plt.tight_layout()
plt.show()


In [ ]:
# 5. محاكاة سطح الحواف ثلاثي الأبعاد بـ OpenCV Sobel
gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
scale = min(90 / gray.shape[0], 90 / gray.shape[1], 1.0)
small = cv2.resize(gray, (max(8, int(gray.shape[1]*scale)), max(8, int(gray.shape[0]*scale))))
gx = cv2.Sobel(small, cv2.CV_32F, 1, 0, ksize=3)
gy = cv2.Sobel(small, cv2.CV_32F, 0, 1, ksize=3)
magnitude = cv2.magnitude(gx, gy)

fig = go.Figure(data=[go.Surface(z=magnitude, colorscale="Viridis")])
fig.update_layout(title="OpenCV Sobel — 3D Edge Magnitude Surface", height=500)
fig.show()
